In [ ]:
import sys
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.svm import SVC
from code.Data_Preprocessing import DataPreprocessing_melt as mf
from code.Data_Processing import TargetVariables as tv
from code.Data_Processing import DataProcessing_run as rf
from joblib import Parallel, delayed
from joblib import Memory

In [ ]:
encoder = LabelEncoder()
MAX_HDRS = 69 # 0-69
MAX_CDI = 90  # CDI: 40-90
lower_pct = 30
upper_pct = 70
# Load MFCC35.pkl
df = pd.read_pickle("/planilhas/MFCC35.pkl")
print(f"Patients: {df['Patient_ID'].nunique()}")

- Note: For calculating the standardized Y values using HDRS/CDI raw score data, refer to the worksheet in the data folder of this project's main DOI. [Use "Patient_ID" as a cross-reference.]

M1 - YAA (S3) (Y = Delta_Y : Worse, Stable and Better)

In [ ]:
df_model_YAA_S3_M = df.copy()
df_model_YAA_S3_M = tv.standardized_classify_evolution(df_model_YAA_S3_M, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_YAA_S3_M['Y_Classe_Evolution_Delta_Y'] = df_model_YAA_S3_M['Y_Classe_Evolution_Delta_Y'].map({'Worse': 0, 'Stable': 1, 'Better': 2})
df_model_YAA_S3_M = df_model_YAA_S3_M.dropna(subset=['Y_Classe_Evolution_Delta_Y'])
df_model_YAA_S3_M['Y_Classe_Evolution_Delta_Y'] = df_model_YAA_S3_M['Y_Classe_Evolution_Delta_Y'].astype(int)

In [ ]:
meta_colsYAA = ['Patient_ID', 'Y_Classe_Evolution_Delta_Y']
df_model_YAA_S3_M = mf.get_mfccs_per_segment_mean_std(df_model_YAA_S3_M, meta_colsYAA)
df_model_YAA_S3_M = df_model_YAA_S3_M.dropna()
print(f"Patients: {df_model_YAA_S3_M['Patient_ID'].nunique()}")

M3 - YA (S3) (Y = Delta_HDRS : Worse, Stable and Better)

In [ ]:
df_model_YA_S3_M = df.copy()
df_model_YA_S3_M = tv.standardized_classify_evolution_HDRS(df_model_YA_S3_M, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_YA_S3_M['Y_Classe_Evolution_HDRS'] = df_model_YA_S3_M['Y_Classe_Evolution_HDRS'].map({'Worse': 0, 'Stable': 1, 'Better': 2})
df_model_YA_S3_M = df_model_YA_S3_M.dropna(subset=['Y_Classe_Evolution_HDRS'])
df_model_YA_S3_M['Y_Classe_Evolution_HDRS'] = df_model_YA_S3_M['Y_Classe_Evolution_HDRS'].astype(int)

In [ ]:
meta_colsYA = ['Patient_ID', 'Y_Classe_Evolution_HDRS']
df_model_YA_S3_M = mf.get_mfccs_per_segment_mean_std(df_model_YA_S3_M, meta_colsYA)
df_model_YA_S3_M = df_model_YA_S3_M.dropna()
print(f"Patients: {df_model_YA_S3_M['Patient_ID'].nunique()}")

M5 - A (S3) (Y = Delta_CDI : Worse, Stable and Better)

In [ ]:
df_model_A_S3_M = df.copy()
df_model_A_S3_M = tv.standardized_classify_evolution_CDI(df_model_A_S3_M, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_A_S3_M['Y_Classe_Evolution_CDI'] = df_model_A_S3_M['Y_Classe_Evolution_CDI'].map({'Worse': 0, 'Stable': 1, 'Better': 2})
df_model_A_S3_M = df_model_A_S3_M.dropna(subset=['Y_Classe_Evolution_CDI'])
df_model_A_S3_M['Y_Classe_Evolution_CDI'] = df_model_A_S3_M['Y_Classe_Evolution_CDI'].astype(int)

In [ ]:
meta_colsA = ['Patient_ID', 'Y_Classe_Evolution_CDI']
df_model_A_S3_M = mf.get_mfccs_per_segment_mean_std(df_model_A_S3_M, meta_colsA)
df_model_A_S3_M = df_model_A_S3_M.dropna()
print(f"Patients: {df_model_A_S3_M['Patient_ID'].nunique()}")

In [ ]:
datasets = [
    (df_model_YAA_S3_M, 'Y_Classe_Evolution_Delta_Y')
    (df_model_YA_S3_M, 'Y_Classe_Evolution_HDRS')
    (df_model_A_S3_M, 'Y_Classe_Evolution_CDI')]

Running the experiments (LOO-CV patient-independent)

In [ ]:
summary_results = []
all_results = []
for idx, (df, target_column) in enumerate(datasets):
    print(f"\nProcessing Dataset {idx+1} - Target: {target_column}")
    unique_patients = df['Patient_ID'].unique()    
    results_rf = [rf.class_process_leave_one_out(patient, df, 'Patient_ID', target_column, RandomForestClassifier(class_weight='balanced', random_state=42), rf.class_param_grid_rf)
        for patient in unique_patients]
    print("RF Done")
    result_lr = [rf.class_process_leave_one_out(patient, df, 'Patient_ID', target_column, LogisticRegression(class_weight='balanced', random_state=42), rf.class_param_grid_logreg)
        for patient in unique_patients]
    print("LogR Done")
    results_xgb = [rf.class_process_leave_one_out(patient, df, 'Patient_ID', target_column, xgb.XGBClassifier(n_jobs=1, random_state=42, use_label_encoder=False, eval_metric='mlogloss', objective='multi:softprob'), rf.class_param_grid_xgb)
        for patient in unique_patients]
    print("XGB Done") 
    results_mlp = [rf.class_process_leave_one_out(patient, df, 'Patient_ID', target_column, MLPClassifier(random_state=42), rf.class_param_grid_mlp)
        for patient in unique_patients]
    print("MLP Done")
    current_results = results_rf + result_lr + results_xgb + results_mlp
    all_results.extend(current_results)   
    df_current = pd.DataFrame([r for r in current_results if r is not None])
    for model_name in df_current["Model"].unique():
        df_model = df_current[df_current["Model"] == model_name]
        summary_results.append({
            "Dataset_Index": idx + 1,
            "Target_Column": target_column,
            "Model": model_name,
            "Recall_Mean": df_model["Recall"].mean(),
            "F1_Score": df_model["F1_Score"].mean(),
            "Precision_Mean": df_model["Precision"].mean()
        })